# 03 — Customer Profiling, Risk Scoring, Health Index and Status Labeling

This notebook is the **bridge between unsupervised learning and supervised machine learning**.

The previous notebook produced the unsupervised segmentation using **K-Means** and **DBSCAN**. Here we use those outputs to create three downstream business layers:

1. **Customer Profiling** — understand who belongs to each K-Means segment and how DBSCAN anomalies are distributed.
2. **Risk Scoring and Customer Health Index** — convert behavioural signals into an interpretable risk/health framework.
3. **Customer Status Labeling** — assign rule-based business labels and create the rule-derived target for the supervised ML notebook.

The notebook is intentionally compact: it keeps only the analyses needed to move from segmentation to supervised modelling, plus the most useful tables and plots for interpretation.

## 0. Imports and notebook settings

We load only the libraries needed for data handling, scoring, validation and visualization. The notebook uses **relative paths**, so it should be placed in the same folder as the CSV exported by `02_KMeans_DBSCAN.ipynb`.

In [ ]:
# ============================================================
# 0. Imports and global settings
# ============================================================

from pathlib import Path
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

# The notebook is designed to be launched from the same folder as the input CSV.
BASE_DIR = Path(".").resolve()
OUTPUT_DIR = BASE_DIR / "outputs_profile_risk_status"
FIGURES_DIR = OUTPUT_DIR / "figures"
TABLES_DIR = OUTPUT_DIR / "tables"

for folder in [OUTPUT_DIR, FIGURES_DIR, TABLES_DIR]:
    folder.mkdir(exist_ok=True)

RANDOM_STATE = 42

def save_current_figure(filename: str) -> None:
    """Save the active Matplotlib figure in the thesis output folder."""
    path = FIGURES_DIR / filename
    plt.savefig(path, dpi=300, bbox_inches="tight")
    print(f"Saved figure: {path}")

print("Setup completed.")
print("Working folder:", BASE_DIR)
print("Output folder:", OUTPUT_DIR)
print("Figures folder:", FIGURES_DIR)
print("Tables folder:", TABLES_DIR)


## 1. Load the clustered dataset

The input must already contain the unsupervised segmentation results from the previous notebook. The expected file is:

`dataset_clustered_kmeans_dbscan_best.csv`

The loader is robust to common CSV separators and encodings, and it also checks a few common output folders.

In [ ]:
# ============================================================
# 1. Robust input loading
# ============================================================

INPUT_CANDIDATES = [
    "dataset_clustered_kmeans_dbscan_best.csv",
    "dataset_clustered_kmeans_dbscan.csv",
    "dataset_kmeans_dbscan.csv",
    "dataset_segmented_kmeans_dbscan.csv",
    "dataset_clustered.csv",
]

SEARCH_DIRS = [
    BASE_DIR,
    BASE_DIR / "outputs_unified_clustering",
    BASE_DIR / "outputs",
]


def read_csv_robust(path: Path) -> pd.DataFrame:
    """Read a CSV file even when separator/encoding differ across environments."""
    encodings = ["utf-8", "utf-8-sig", "latin1", "cp1252"]
    separators = [",", ";", "\t"]
    last_error = None

    for encoding in encodings:
        for sep in separators:
            try:
                candidate = pd.read_csv(path, encoding=encoding, sep=sep)
                if candidate.shape[1] > 1:
                    return candidate
            except Exception as exc:
                last_error = exc

    raise ValueError(f"Could not read {path}. Last error: {last_error}")


def find_input_file() -> Path:
    """Find the clustered output exported by 02_KMeans_DBSCAN.ipynb."""
    checked = []
    for folder in SEARCH_DIRS:
        for filename in INPUT_CANDIDATES:
            path = folder / filename
            checked.append(path)
            if path.exists():
                return path

    checked_text = "\n".join(str(path) for path in checked)
    raise FileNotFoundError(
        "Could not find the clustered dataset. Place this notebook in the same folder as "
        "`dataset_clustered_kmeans_dbscan_best.csv`, or run `02_KMeans_DBSCAN.ipynb` first.\n\n"
        f"Checked paths:\n{checked_text}"
    )


input_path = find_input_file()
df = read_csv_robust(input_path)

print("Loaded file:", input_path)
print("Dataset shape:", df.shape)
display(df.head())

## 2. Prepare and validate the input

This section checks that the segmentation output is usable for downstream analysis. It harmonizes a few column names, detects monthly transaction columns, rebuilds behavioural variables only if needed, and verifies that K-Means labels are present.

**Outcome:** after this section, the dataset is ready for customer profiling, risk scoring and status labeling.

In [ ]:
# ============================================================
# 2. Column harmonization, validation and safety reconstruction
# ============================================================

# Harmonize common Italian/raw names if they are still present.
rename_map = {
    "Natura": "Nature",
    "Fatturato": "Revenue",
    "Dipendenti": "Employees",
    "Provincia": "Province",
    "ATECO_Desc": "NACE_Desc",
}
df = df.rename(columns={old: new for old, new in rename_map.items() if old in df.columns})

# Detect and sort monthly columns from oldest to newest: M-35 ... M-0.
month_cols = [col for col in df.columns if re.fullmatch(r"M-\d+", str(col))]
month_cols = sorted(month_cols, key=lambda x: -int(str(x).split("-")[1]))

# Convert numeric columns safely.
numeric_candidates = [
    "Revenue", "Employees", "total_volume", "recent_volume", "past_volume", "avg_monthly_volume",
    "volume_trend", "trend_pct", "volatility", "cv_volume", "recent_ratio", "activity_drop_ratio",
] + month_cols

for col in numeric_candidates:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce").replace([np.inf, -np.inf], np.nan)

# K-Means cluster labels are mandatory because this notebook profiles the unsupervised solution.
if "kmeans_cluster" not in df.columns:
    raise ValueError(
        "Missing `kmeans_cluster`. Run the previous K-Means/DBSCAN notebook first and use its clustered output."
    )

# If segment names are not available, create neutral labels from cluster numbers.
if "kmeans_segment" not in df.columns:
    df["kmeans_segment"] = "Cluster " + df["kmeans_cluster"].astype(str)

# Recreate DBSCAN flags if only dbscan_label is present.
if "is_dbscan_noise" not in df.columns and "dbscan_label" in df.columns:
    df["is_dbscan_noise"] = df["dbscan_label"].eq(-1)
if "dbscan_group" not in df.columns and "dbscan_label" in df.columns:
    df["dbscan_group"] = np.where(
        df["dbscan_label"].eq(-1),
        "Noise / atypical customer",
        "Dense customer population",
    )
if "is_dbscan_noise" not in df.columns:
    df["is_dbscan_noise"] = False

# Essential behavioural variables used in profiling, scoring and status labeling.
required_behavioural = [
    "total_volume", "recent_volume", "past_volume", "avg_monthly_volume",
    "volume_trend", "trend_pct", "volatility", "cv_volume",
    "recent_ratio", "activity_drop_ratio",
]

missing_behavioural = [col for col in required_behavioural if col not in df.columns]

# Reconstruct missing behavioural variables from the monthly history if needed.
if missing_behavioural and month_cols:
    monthly = df[month_cols].fillna(0)
    recent_cols = [c for c in ["M-5", "M-4", "M-3", "M-2", "M-1", "M-0"] if c in df.columns]
    past_cols = [c for c in month_cols if c not in recent_cols]

    df["total_volume"] = monthly.sum(axis=1)
    df["avg_monthly_volume"] = monthly.mean(axis=1)
    df["recent_volume"] = df[recent_cols].fillna(0).sum(axis=1) if recent_cols else 0
    df["past_volume"] = df[past_cols].fillna(0).sum(axis=1) if past_cols else 0

    recent_months = max(len(recent_cols), 1)
    past_months = max(len(past_cols), 1)
    recent_avg = df["recent_volume"] / recent_months
    past_avg = df["past_volume"] / past_months

    df["volume_trend"] = recent_avg - past_avg
    df["trend_pct"] = (recent_avg - past_avg) / (past_avg.abs() + 1)
    df["volatility"] = monthly.std(axis=1)
    df["cv_volume"] = df["volatility"] / (df["avg_monthly_volume"].abs() + 1)
    df["recent_ratio"] = df["recent_volume"] / (df["total_volume"].abs() + 1)

    expected_recent = past_avg * recent_months
    df["activity_drop_ratio"] = (expected_recent - df["recent_volume"]) / (expected_recent.abs() + 1)

missing_after = [col for col in required_behavioural if col not in df.columns]
if missing_after:
    raise ValueError(
        "Missing behavioural variables needed for downstream analysis: "
        f"{missing_after}. Use the full output from the clustering notebook."
    )

# Final numeric cleanup for scoring stability.
for col in required_behavioural:
    df[col] = pd.to_numeric(df[col], errors="coerce").replace([np.inf, -np.inf], np.nan)
    df[col] = df[col].fillna(df[col].median())

# ------------------------------------------------------------
# Unit-of-analysis audit
# ------------------------------------------------------------
# This notebook expects one row per customer/customer account after the previous clustering step.
# If a unique customer identifier exists, the audit reports whether duplicate customer rows are present.
possible_customer_id_columns = [
    "customer_id", "Customer_ID", "CustomerID", "Codice_Cliente", "client_id", "Cliente",
    "customer_code", "CustomerCode", "id_cliente", "ID_cliente", "ID_Cliente",
]
customer_id_col = next((col for col in possible_customer_id_columns if col in df.columns), None)

if customer_id_col is not None:
    unique_customers = df[customer_id_col].nunique(dropna=True)
    duplicate_customer_rows = int(df.duplicated(subset=customer_id_col).sum())
    unit_of_analysis = f"Customer-level after checking `{customer_id_col}`"
else:
    unique_customers = len(df)
    duplicate_customer_rows = 0
    unit_of_analysis = "No explicit customer ID detected; each row is treated as one customer record"

readiness = pd.DataFrame({
    "item": [
        "Rows/customer records",
        "Unique customers used for analysis",
        "Duplicate customer rows detected",
        "Unit-of-analysis assumption",
        "K-Means segments",
        "Monthly columns",
        "DBSCAN noise share",
        "Behavioural variables",
    ],
    "value": [
        len(df),
        unique_customers,
        duplicate_customer_rows,
        unit_of_analysis,
        df["kmeans_cluster"].nunique(),
        len(month_cols),
        f"{df['is_dbscan_noise'].mean() * 100:.2f}%",
        f"{len(required_behavioural)}/{len(required_behavioural)} available",
    ],
})

display(readiness)
display(Markdown(
    f"**Outcome.** The notebook will analyze **{unique_customers:,} customer records**. "
    f"The dataset contains **{df['kmeans_cluster'].nunique()} K-Means segments**, and the DBSCAN noise share is "
    f"**{df['is_dbscan_noise'].mean() * 100:.2f}%**. The behavioural variables required for downstream scoring are ready."
))

# Important consistency check for thesis tables/figures.
if unique_customers != len(df):
    display(Markdown(
        "**Warning.** Duplicate customer identifiers were detected. "
        "Review whether the input is customer-level or transactional/monthly-level before using the figures in the thesis."
    ))


# Part A — Customer Profiling

The objective is not to repeat EDA. The objective is to summarize the segmentation output in a way that can support risk scoring and business interpretation.

We profile each K-Means segment by size, activity, trend, volatility, recent engagement, DBSCAN anomaly share, and key categorical descriptors.

In [ ]:
# ============================================================
# 3. Compact K-Means segment profile
# ============================================================

profile_cols = [
    "Revenue", "Employees", "total_volume", "recent_volume", "avg_monthly_volume",
    "trend_pct", "cv_volume", "recent_ratio", "activity_drop_ratio",
]
profile_cols = [col for col in profile_cols if col in df.columns]

profile_aggs = {
    "customers": ("kmeans_cluster", "size"),
    "dbscan_noise_share_%": ("is_dbscan_noise", lambda x: x.mean() * 100),
}
for col in profile_cols:
    profile_aggs[f"avg_{col}"] = (col, "mean")

segment_profile = (
    df.groupby(["kmeans_cluster", "kmeans_segment"])
      .agg(**profile_aggs)
      .reset_index()
      .sort_values("kmeans_cluster")
)
segment_profile["customer_share_%"] = segment_profile["customers"] / len(df) * 100

# Add the most frequent Region / NACE / Nature where available.
def top_category(series):
    clean = series.dropna().astype(str)
    return clean.value_counts().idxmax() if len(clean) else "Not available"

for cat_col, out_col in [
    ("Region", "top_region"),
    ("NACE_Desc", "top_sector"),
    ("Nature", "top_legal_nature"),
]:
    if cat_col in df.columns:
        top_values = df.groupby(["kmeans_cluster", "kmeans_segment"])[cat_col].apply(top_category).reset_index(name=out_col)
        segment_profile = segment_profile.merge(top_values, on=["kmeans_cluster", "kmeans_segment"], how="left")

# Reorder the most interpretable columns first.
front_cols = [
    "kmeans_cluster", "kmeans_segment", "customers", "customer_share_%", "dbscan_noise_share_%",
    "avg_total_volume", "avg_recent_volume", "avg_trend_pct", "avg_recent_ratio",
    "avg_activity_drop_ratio", "avg_cv_volume", "avg_Revenue", "avg_Employees",
]
front_cols = [col for col in front_cols if col in segment_profile.columns]
remaining_cols = [col for col in segment_profile.columns if col not in front_cols]
segment_profile = segment_profile[front_cols + remaining_cols]

print("Segment profile — compact business view:")
display(segment_profile.round(3))

largest_segment = segment_profile.sort_values("customers", ascending=False).iloc[0]
highest_noise_segment = segment_profile.sort_values("dbscan_noise_share_%", ascending=False).iloc[0]

display(Markdown(
    f"**Outcome.** The largest segment is **{largest_segment['kmeans_segment']}** "
    f"({largest_segment['customers']:,} customers). The highest DBSCAN anomaly concentration is in "
    f"**{highest_noise_segment['kmeans_segment']}** "
    f"({highest_noise_segment['dbscan_noise_share_%']:.2f}% noise share)."
))

In [ ]:
# ============================================================
# 4. Essential profiling visualizations
# ============================================================

# 1) Segment size: shows the commercial weight of each segment.
plot_df = segment_profile.sort_values("customers")
plt.figure(figsize=(10, 5))
plt.barh(plot_df["kmeans_segment"], plot_df["customers"])
plt.title("Customer count by K-Means segment")
plt.xlabel("Number of customers")
plt.ylabel("K-Means segment")
plt.tight_layout()
plt.show()

# 2) DBSCAN noise share by segment: shows where atypical customers concentrate.
plot_df = segment_profile.sort_values("dbscan_noise_share_%")
plt.figure(figsize=(10, 5))
plt.barh(plot_df["kmeans_segment"], plot_df["dbscan_noise_share_%"])
plt.title("DBSCAN anomaly share by K-Means segment")
plt.xlabel("DBSCAN noise share (%)")
plt.ylabel("K-Means segment")
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 5. Segment heatmap: behavioural profile versus portfolio average
# ============================================================

# The heatmap uses standardized segment averages. Positive values mean above-average segment behaviour;
# negative values mean below-average segment behaviour.
heatmap_vars = [
    "total_volume", "recent_volume", "trend_pct", "recent_ratio",
    "activity_drop_ratio", "cv_volume", "Revenue", "Employees",
]
heatmap_vars = [col for col in heatmap_vars if col in df.columns]

segment_means = df.groupby("kmeans_segment")[heatmap_vars].mean()
overall_mean = df[heatmap_vars].mean()
overall_std = df[heatmap_vars].std().replace(0, np.nan)
standardized_segment_means = ((segment_means - overall_mean) / overall_std).fillna(0)

fig, ax = plt.subplots(figsize=(12, max(4, 0.6 * len(standardized_segment_means))))
im = ax.imshow(standardized_segment_means.values, aspect="auto")
ax.set_title("Standardized behavioural profile by K-Means segment")
ax.set_xticks(range(len(heatmap_vars)))
ax.set_xticklabels(heatmap_vars, rotation=45, ha="right")
ax.set_yticks(range(len(standardized_segment_means.index)))
ax.set_yticklabels(standardized_segment_means.index)
fig.colorbar(im, ax=ax, label="Standard deviations from portfolio average")
plt.tight_layout()
plt.show()

display(Markdown(
    "**Outcome.** This heatmap is the key visual bridge from clustering to business interpretation: "
    "it shows which segments are above or below the portfolio average in value, recent activity, trend, volatility and decline signals."
))

# Part B — Customer Risk Score and Customer Health Index

The risk framework converts behavioural evidence into an interpretable score.

The risk score uses six thesis-aligned components:

- low recent activity;
- activity decline/drop;
- negative trend;
- high volatility;
- DBSCAN anomaly status;
- K-Means segment context.

The **Customer Risk Score** ranges from 0 to 100. The **Customer Health Index** is its inverse: `100 - risk score`.

In [ ]:
# ============================================================
# 6. Scoring helpers and thesis-aligned weights
# ============================================================

def percentile_score(series, higher_means_higher_score=True):
    """
    Convert a variable into a 0-100 percentile score.

    This makes variables with different units comparable and limits the influence of outliers.
    """
    s = pd.to_numeric(series, errors="coerce").replace([np.inf, -np.inf], np.nan)
    if s.notna().sum() == 0:
        return pd.Series(50.0, index=series.index)

    filled = s.fillna(s.median())
    if filled.nunique(dropna=False) <= 1:
        return pd.Series(50.0, index=series.index)

    score = filled.rank(method="average", pct=True) * 100
    if not higher_means_higher_score:
        score = 100 - score
    return score.clip(0, 100)


SCORING_WEIGHTS = {
    "recent_activity_risk": 0.25,
    "activity_drop_risk": 0.20,
    "trend_risk": 0.20,
    "volatility_risk": 0.15,
    "dbscan_anomaly_risk": 0.10,
    "kmeans_segment_context_risk": 0.10,
}

weights_df = pd.DataFrame({
    "component": list(SCORING_WEIGHTS.keys()),
    "weight": list(SCORING_WEIGHTS.values()),
})
weights_df["weight_%"] = weights_df["weight"] * 100

print("Customer Risk Score components:")
display(weights_df)

In [ ]:
# ============================================================
# 7. Compute Customer Risk Score, Health Index and risk classes
# ============================================================

# Recent activity: lower recent volume means higher risk.
df["recent_activity_risk"] = percentile_score(
    np.log1p(df["recent_volume"].clip(lower=0)),
    higher_means_higher_score=False,
)

# Activity drop: higher drop means higher risk.
df["activity_drop_risk"] = percentile_score(
    df["activity_drop_ratio"],
    higher_means_higher_score=True,
)

# Trend: lower/negative trend means higher risk.
df["trend_risk"] = percentile_score(
    df["trend_pct"],
    higher_means_higher_score=False,
)

# Volatility: more unstable behaviour means higher risk.
df["volatility_risk"] = percentile_score(
    df["cv_volume"],
    higher_means_higher_score=True,
)

# DBSCAN anomaly: noise points receive high anomaly risk.
df["dbscan_anomaly_risk"] = np.where(df["is_dbscan_noise"].astype(bool), 100.0, 0.0)

# K-Means context: clusters with higher average behavioural risk receive higher context risk.
base_risk_components = [
    "recent_activity_risk", "activity_drop_risk", "trend_risk", "volatility_risk", "dbscan_anomaly_risk",
]
df["base_behavioural_risk"] = df[base_risk_components].mean(axis=1)
segment_base_risk = df.groupby("kmeans_cluster")["base_behavioural_risk"].mean()
segment_context_map = percentile_score(segment_base_risk, higher_means_higher_score=True)
df["kmeans_segment_context_risk"] = df["kmeans_cluster"].map(segment_context_map).fillna(50.0)

# Weighted final score.
df["customer_risk_score"] = sum(df[col] * weight for col, weight in SCORING_WEIGHTS.items()).clip(0, 100).round(2)
df["customer_health_index"] = (100 - df["customer_risk_score"]).clip(0, 100).round(2)
df["predicted_churn_risk_probability"] = (df["customer_risk_score"] / 100).round(4)

# Three thesis risk classes: Low, Medium, High.
# The split is percentile-based, so it adapts to the portfolio and avoids arbitrary absolute thresholds.
REFERENCE_CUSTOMERS = 16791
LOW_RISK_SHARE = 7668 / REFERENCE_CUSTOMERS
MEDIUM_RISK_SHARE = 5526 / REFERENCE_CUSTOMERS
LOW_MEDIUM_CUM_SHARE = LOW_RISK_SHARE + MEDIUM_RISK_SHARE

risk_rank = df["customer_risk_score"].rank(method="first", pct=True)
df["customer_risk_class"] = np.select(
    [risk_rank <= LOW_RISK_SHARE, risk_rank <= LOW_MEDIUM_CUM_SHARE],
    ["Low Risk", "Medium Risk"],
    default="High Risk",
)

# Value is not part of customer health. It is used only to prioritize business action.
df["value_score"] = percentile_score(np.log1p(df["total_volume"].clip(lower=0)), higher_means_higher_score=True)
df["value_at_risk_score"] = (df["value_score"] * df["predicted_churn_risk_probability"]).round(2)

# Explainable flags: these show why customers are risky.
df["flag_negative_trend"] = df["trend_pct"] <= df["trend_pct"].quantile(0.25)
df["flag_high_activity_drop"] = df["activity_drop_ratio"] >= df["activity_drop_ratio"].quantile(0.75)
df["flag_low_recent_activity"] = df["recent_volume"] <= df["recent_volume"].quantile(0.25)
df["flag_high_volatility"] = df["cv_volume"] >= df["cv_volume"].quantile(0.75)
df["flag_dbscan_noise"] = df["is_dbscan_noise"].astype(bool)
df["flag_high_value"] = df["value_score"] >= df["value_score"].quantile(0.75)

risk_flag_cols = [
    "flag_negative_trend", "flag_high_activity_drop", "flag_low_recent_activity",
    "flag_high_volatility", "flag_dbscan_noise",
]
df["risk_driver_count"] = df[risk_flag_cols].sum(axis=1)
df["strategic_at_risk_flag"] = df["flag_high_value"] & df["customer_risk_class"].eq("High Risk")

risk_distribution = df["customer_risk_class"].value_counts().rename_axis("customer_risk_class").reset_index(name="customers")
risk_distribution["share_%"] = risk_distribution["customers"] / len(df) * 100

print("Risk class distribution:")
display(risk_distribution.round(2))

print("Risk/health score summary:")
display(df[["customer_risk_score", "customer_health_index", "value_at_risk_score", "risk_driver_count"]].describe().T.round(2))

display(Markdown(
    f"**Outcome.** The risk framework identifies **{(df['customer_risk_class'] == 'High Risk').sum():,} high-risk customers**. "
    f"Among them, **{df['strategic_at_risk_flag'].sum():,} are also high-value customers**, so they should be prioritized for account review."
))

In [ ]:
# ============================================================
# 8. Thesis Fig. 9 and Tab. XI — risk visualizations by K-Means segment
# ============================================================
# This cell generates the corrected thesis figures and tables for Section 5.2.
# It uses the current dataframe `df`, i.e. the same customer-level dataset used for
# K-Means, DBSCAN, risk scoring and status labeling.

# ------------------------------------------------------------
# 8.1 Safety check: confirm the data used for Fig. 9 / Tab. XI
# ------------------------------------------------------------
print("Data used for risk visualizations:")
print(f"Rows/customer records in df: {len(df):,}")
print("K-Means segment counts:")
display(
    df["kmeans_segment"]
    .value_counts()
    .rename_axis("kmeans_segment")
    .reset_index(name="customers")
)

# ------------------------------------------------------------
# 8.2 Overall Customer Health Index distribution
# ------------------------------------------------------------
fig, ax = plt.subplots(figsize=(9, 5), dpi=200)
ax.hist(df["customer_health_index"], bins=30)
ax.set_title("Distribution of Customer Health Index")
ax.set_xlabel("Customer Health Index")
ax.set_ylabel("Number of customers")
plt.tight_layout()
save_current_figure("fig_customer_health_index_distribution.png")
plt.show()

# ------------------------------------------------------------
# 8.3 Tab. XI — corrected risk summary by K-Means segment
# ------------------------------------------------------------
risk_summary_by_segment = (
    df.groupby(["kmeans_cluster", "kmeans_segment"])
      .agg(
          customers=("kmeans_cluster", "size"),
          avg_health_index=("customer_health_index", "mean"),
          avg_risk_score=("customer_risk_score", "mean"),
          high_risk_share=("customer_risk_class", lambda x: (x == "High Risk").mean() * 100),
          strategic_at_risk_customers=("strategic_at_risk_flag", "sum"),
          avg_value_at_risk_score=("value_at_risk_score", "mean"),
      )
      .reset_index()
)

risk_summary_by_segment["customer_share_pct"] = risk_summary_by_segment["customers"] / len(df) * 100

# Thesis order: highest average risk first. This order matches the discussion in Section 5.2.
risk_summary_by_segment = risk_summary_by_segment.sort_values("avg_risk_score", ascending=False).reset_index(drop=True)

# A thesis-ready table with clear labels and rounded values.
tab_xi_risk_summary = risk_summary_by_segment.rename(columns={
    "kmeans_cluster": "K-Means cluster",
    "kmeans_segment": "K-Means segment",
    "customers": "Customers (#)",
    "customer_share_pct": "Customer share (%)",
    "avg_health_index": "Avg. Health Index",
    "avg_risk_score": "Avg. Risk Score",
    "high_risk_share": "High-Risk Share (%)",
    "strategic_at_risk_customers": "Strategic at-risk customers (#)",
    "avg_value_at_risk_score": "Avg. Value-at-Risk Score",
})

# Reorder columns for thesis export.
tab_xi_risk_summary = tab_xi_risk_summary[[
    "K-Means segment",
    "K-Means cluster",
    "Customers (#)",
    "Customer share (%)",
    "Avg. Health Index",
    "Avg. Risk Score",
    "High-Risk Share (%)",
    "Strategic at-risk customers (#)",
    "Avg. Value-at-Risk Score",
]]

print("Tab. XI — corrected risk summary by K-Means segment:")
display(tab_xi_risk_summary.round(2))

tab_xi_risk_summary.round(2).to_csv(TABLES_DIR / "tab_xi_risk_summary_by_kmeans_segment.csv", index=False)

# ------------------------------------------------------------
# 8.4 Corrected Fig. 9 — direct replacement for the current thesis Fig. 9
# ------------------------------------------------------------
# This reproduces the same logic as the current document figure, but with the correct 16,791-row dataset.
# Use this image if you want to replace the current Fig. 9 without changing the figure type.

plot_risk = risk_summary_by_segment.sort_values("high_risk_share", ascending=True)
plot_health = risk_summary_by_segment.sort_values("avg_health_index", ascending=True)

fig, axes = plt.subplots(2, 1, figsize=(9, 8), dpi=200)

bars_risk = axes[0].barh(plot_risk["kmeans_segment"], plot_risk["high_risk_share"])
axes[0].set_title("High-risk share by K-Means segment")
axes[0].set_xlabel("High-risk customers (%)")
axes[0].set_ylabel("")
max_risk = max(plot_risk["high_risk_share"].max(), 1)
for bar in bars_risk:
    width = bar.get_width()
    axes[0].text(width + max_risk * 0.01, bar.get_y() + bar.get_height() / 2, f"{width:.1f}%", va="center", fontsize=8)

bars_health = axes[1].barh(plot_health["kmeans_segment"], plot_health["avg_health_index"])
axes[1].set_title("Average Customer Health Index by K-Means segment")
axes[1].set_xlabel("Average Customer Health Index")
axes[1].set_ylabel("")
max_health = max(plot_health["avg_health_index"].max(), 1)
for bar in bars_health:
    width = bar.get_width()
    axes[1].text(width + max_health * 0.01, bar.get_y() + bar.get_height() / 2, f"{width:.1f}", va="center", fontsize=8)

plt.tight_layout()
save_current_figure("fig_09_risk_summary_by_kmeans_segment.png")
plt.show()

# ------------------------------------------------------------
# 8.5 Optional Fig. 9b — risk-class composition by K-Means segment
# ------------------------------------------------------------
# This optional figure is more detailed than the two-panel figure above because it shows the
# internal Low/Medium/High Risk composition of each segment.

risk_class_order = ["Low Risk", "Medium Risk", "High Risk"]
segment_order = risk_summary_by_segment["kmeans_segment"].tolist()

risk_mix_counts = pd.crosstab(
    df["kmeans_segment"],
    df["customer_risk_class"],
).reindex(index=segment_order, columns=risk_class_order, fill_value=0)

risk_mix = (
    pd.crosstab(df["kmeans_segment"], df["customer_risk_class"], normalize="index") * 100
).reindex(index=segment_order, columns=risk_class_order, fill_value=0)

print("Risk-class composition by K-Means segment — counts:")
display(risk_mix_counts)

print("Risk-class composition by K-Means segment — share (%):")
display(risk_mix.round(2))

ax = risk_mix.plot(kind="bar", stacked=True, figsize=(11, 5), width=0.75)
ax.set_title("Risk-class composition by K-Means segment")
ax.set_xlabel("K-Means segment")
ax.set_ylabel("Share of customers (%)")
plt.xticks(rotation=30, ha="right")
plt.legend(title="Risk class", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
save_current_figure("fig_09b_risk_class_composition_by_kmeans_segment.png")
plt.show()

risk_mix_counts.to_csv(TABLES_DIR / "risk_class_composition_by_kmeans_segment_counts.csv")
risk_mix.round(2).to_csv(TABLES_DIR / "risk_class_composition_by_kmeans_segment_share_pct.csv")


# Part C — Customer Status Labeling and supervised target

The status labels translate risk and activity patterns into business-readable categories. These labels also create the **rule-derived churn-risk target** for the supervised ML notebook.

Important limitation: `churn_risk_target` is **not observed future churn**. It is a transparent rule-derived label based on customer behaviour, segment context and anomaly status.

All counts in this section must refer to the same unit of analysis used throughout the notebook: **one customer/customer record per row**. Therefore, the status table and the status figure are recalculated directly from `df` to avoid inconsistencies between old variables and updated plots.


In [ ]:
# ============================================================
# 9. Activity indicators for status labeling
# ============================================================

if month_cols:
    monthly = df[month_cols].fillna(0)
    activity = monthly.gt(0)

    recent_cols = [c for c in ["M-5", "M-4", "M-3", "M-2", "M-1", "M-0"] if c in df.columns]
    historical_cols = [c for c in month_cols if c not in recent_cols]

    df["total_active_months"] = activity.sum(axis=1)
    df["recent_active_months"] = activity[recent_cols].sum(axis=1) if recent_cols else 0
    df["historical_active_months"] = activity[historical_cols].sum(axis=1) if historical_cols else 0
    df["recent_status_volume"] = monthly[recent_cols].sum(axis=1) if recent_cols else df["recent_volume"]
    df["historical_status_volume"] = monthly[historical_cols].sum(axis=1) if historical_cols else df["past_volume"]

    # Vectorized months-since-last-activity. 0 = active in M-0, 1 = last active in M-1, etc.
    offsets = np.array([int(col.split("-")[1]) for col in month_cols])
    last_activity = np.where(activity.to_numpy(), offsets, np.inf).min(axis=1)
    df["months_since_last_activity"] = np.where(np.isinf(last_activity), np.nan, last_activity)
else:
    # Fallback if monthly history is unavailable.
    df["total_active_months"] = np.where(df["total_volume"] > 0, 1, 0)
    df["recent_active_months"] = np.where(df["recent_volume"] > 0, 1, 0)
    df["historical_active_months"] = np.where(df["past_volume"] > 0, 1, 0)
    df["recent_status_volume"] = df["recent_volume"]
    df["historical_status_volume"] = df["past_volume"]
    df["months_since_last_activity"] = np.where(df["recent_volume"] > 0, 0, np.nan)

activity_summary_cols = [
    "total_active_months", "recent_active_months", "historical_active_months",
    "recent_status_volume", "historical_status_volume", "months_since_last_activity",
]

print("Activity indicator summary:")
display(df[activity_summary_cols].describe().T.round(2))

In [ ]:
# ============================================================
# 10. Thesis-aligned customer status labels and churn-risk target
# ============================================================

# Data-driven thresholds for growth, decline and volatility.
growth_trend_threshold = df["trend_pct"].quantile(0.75)
decline_trend_threshold = df["trend_pct"].quantile(0.25)
high_drop_threshold = df["activity_drop_ratio"].quantile(0.75)
high_volatility_threshold = df["cv_volume"].quantile(0.75)
engagement_median = df["recent_ratio"].median()

recent_active = df["recent_active_months"] > 0
historical_active = df["historical_active_months"] > 0
high_risk = df["customer_risk_class"].eq("High Risk")
low_risk = df["customer_risk_class"].eq("Low Risk")

is_anomalous_high_risk = df["flag_dbscan_noise"] & high_risk
is_lost_or_inactive = (~recent_active) & (historical_active | (df["total_volume"] <= 0))

declining_signal = (df["trend_pct"] <= decline_trend_threshold) | (df["activity_drop_ratio"] >= high_drop_threshold)
is_declining = recent_active & high_risk & declining_signal

volatile_signal = df["cv_volume"] >= high_volatility_threshold
is_volatile_monitor = recent_active & volatile_signal & (~high_risk)

growth_signal = (
    recent_active &
    (df["trend_pct"] >= growth_trend_threshold) &
    (df["recent_ratio"] >= engagement_median)
)

is_stable_healthy = recent_active & low_risk & (~declining_signal)

conditions = [
    is_anomalous_high_risk,
    is_lost_or_inactive,
    is_declining,
    is_volatile_monitor,
    growth_signal,
    is_stable_healthy,
]
labels = [
    "Anomalous_High_Risk",
    "Lost_or_Inactive",
    "Declining",
    "Volatile_Monitor",
    "Active_Growth",
    "Stable_Healthy",
]

df["customer_status_label"] = np.select(conditions, labels, default="Unclassified_Check")

# Positive class for supervised learning: customers that are lost/inactive, declining, or anomalous high risk.
positive_statuses = {"Anomalous_High_Risk", "Lost_or_Inactive", "Declining"}
df["churn_risk_target"] = df["customer_status_label"].isin(positive_statuses).astype(int)

# Priority and action fields are useful for business interpretation but should not be used as ML predictors.
status_priority_map = {
    "Lost_or_Inactive": 0,
    "Anomalous_High_Risk": 1,
    "Declining": 2,
    "Volatile_Monitor": 3,
    "Unclassified_Check": 4,
    "Stable_Healthy": 5,
    "Active_Growth": 6,
}
df["status_priority"] = df["customer_status_label"].map(status_priority_map).fillna(4).astype(int)

conditions_action = [
    df["strategic_at_risk_flag"],
    df["customer_status_label"].eq("Anomalous_High_Risk"),
    df["customer_status_label"].eq("Lost_or_Inactive"),
    df["customer_status_label"].eq("Declining"),
    df["customer_status_label"].eq("Volatile_Monitor"),
    df["customer_status_label"].eq("Active_Growth"),
    df["customer_status_label"].eq("Stable_Healthy"),
]
actions = [
    "Immediate account review: high-value customer with elevated risk.",
    "Priority manual review: distinguish anomaly, opportunity and deterioration.",
    "Win-back campaign: investigate inactivity and test reactivation offer.",
    "Retention intervention: contact customer and diagnose decline drivers.",
    "Monitoring workflow: check seasonality, service issues or irregular demand.",
    "Growth action: upsell/cross-sell and secure recurring volume.",
    "Maintain efficiently: periodic check-in and service-quality monitoring.",
]
df["recommended_action"] = np.select(conditions_action, actions, default="Manual data/business check before action.")

# Business priority helps rank customers for action. It is not a supervised-learning feature.
status_priority_scaled = df["status_priority"] / max(df["status_priority"].max(), 1)
df["business_priority_score"] = (
    0.45 * df["customer_risk_score"] +
    0.35 * df["value_score"] +
    0.20 * (100 * (1 - status_priority_scaled)) +
    np.where(df["customer_status_label"].eq("Anomalous_High_Risk"), 15, 0)
).clip(0, 100).round(2)

# Recalculate distributions directly from the final dataframe used by this notebook.
status_order = [
    "Lost_or_Inactive",
    "Anomalous_High_Risk",
    "Declining",
    "Volatile_Monitor",
    "Unclassified_Check",
    "Stable_Healthy",
    "Active_Growth",
]

status_distribution = (
    df["customer_status_label"]
    .value_counts()
    .reindex(status_order, fill_value=0)
    .rename_axis("customer_status_label")
    .reset_index(name="customers")
)
status_distribution["share_%"] = (status_distribution["customers"] / len(df) * 100).round(2)
status_distribution = status_distribution.sort_values("customers", ascending=False).reset_index(drop=True)

target_distribution = (
    df["churn_risk_target"]
    .value_counts()
    .sort_index()
    .rename_axis("churn_risk_target")
    .reset_index(name="customers")
)
target_distribution["share_%"] = (target_distribution["customers"] / len(df) * 100).round(2)
target_distribution["target_label"] = target_distribution["churn_risk_target"].map({
    0: "Not rule-derived churn risk",
    1: "Rule-derived churn risk",
})

print("Customer status distribution:")
display(status_distribution)

print("Rule-derived churn-risk target distribution:")
display(target_distribution)

print(f"Total customers/status labels counted: {int(status_distribution['customers'].sum()):,}")
print(f"Rows in df: {len(df):,}")

display(Markdown(
    f"**Outcome.** The supervised target flags **{df['churn_risk_target'].sum():,} customers** "
    f"out of **{len(df):,} customer records** as rule-derived churn risk. "
    "This target will be used in the next notebook to train and compare supervised models."
))


In [ ]:
# ============================================================
# 11. Status visualizations and target readiness
# ============================================================

# Recompute directly from df to avoid using stale variables from previous notebook runs.
status_distribution = (
    df["customer_status_label"]
    .value_counts()
    .rename_axis("customer_status_label")
    .reset_index(name="customers")
)
status_distribution["share_%"] = (status_distribution["customers"] / len(df) * 100).round(2)

print("Status distribution used for the plot:")
display(status_distribution)
print(f"Total customers plotted: {int(status_distribution['customers'].sum()):,}")
print(f"Rows in df: {len(df):,}")

# 1) Customer status distribution.
plot_status = status_distribution.sort_values("customers", ascending=True)

fig, ax = plt.subplots(figsize=(9, 5), dpi=200)
bars = ax.barh(plot_status["customer_status_label"], plot_status["customers"])
ax.set_title("Customer status label distribution")
ax.set_xlabel("Number of customers")
ax.set_ylabel("Customer status")

max_value = plot_status["customers"].max()
for bar in bars:
    width = bar.get_width()
    ax.text(
        width + max_value * 0.01,
        bar.get_y() + bar.get_height() / 2,
        f"{int(width):,}",
        va="center",
        fontsize=8,
    )

plt.tight_layout()
save_current_figure("fig_customer_status_label_distribution.png")
plt.show()

# 2) Churn-risk target rate by K-Means segment.
target_by_segment = (
    df.groupby("kmeans_segment")
      .agg(
          customers=("churn_risk_target", "count"),
          risky_customers=("churn_risk_target", "sum"),
          target_rate_pct=("churn_risk_target", lambda x: x.mean() * 100),
      )
      .reset_index()
      .sort_values("target_rate_pct", ascending=True)
)

print("Target rate by segment:")
display(target_by_segment.round(2))

fig, ax = plt.subplots(figsize=(9, 5), dpi=200)
bars = ax.barh(target_by_segment["kmeans_segment"], target_by_segment["target_rate_pct"])
ax.set_title("Rule-derived churn-risk target rate by K-Means segment")
ax.set_xlabel("Target rate (%)")
ax.set_ylabel("K-Means segment")

max_rate = max(target_by_segment["target_rate_pct"].max(), 1)
for bar in bars:
    width = bar.get_width()
    ax.text(
        width + max_rate * 0.01,
        bar.get_y() + bar.get_height() / 2,
        f"{width:.1f}%",
        va="center",
        fontsize=8,
    )

plt.tight_layout()
save_current_figure("fig_churn_risk_target_rate_by_kmeans_segment.png")
plt.show()

# 3) Thesis-style two-panel figure for Section 5.3.
fig, axes = plt.subplots(1, 2, figsize=(14, 5), dpi=200)

axes[0].barh(plot_status["customer_status_label"], plot_status["customers"])
axes[0].set_title("Customer status label distribution")
axes[0].set_xlabel("Number of customers")
axes[0].set_ylabel("Customer status")

axes[1].barh(target_by_segment["kmeans_segment"], target_by_segment["target_rate_pct"])
axes[1].set_title("Rule-derived churn-risk target rate by K-Means segment")
axes[1].set_xlabel("Target rate (%)")
axes[1].set_ylabel("K-Means segment")

plt.tight_layout()
save_current_figure("fig_status_labels_and_target_readiness.png")
plt.show()

# Export the tables behind the figures.
status_distribution.to_csv(TABLES_DIR / "customer_status_distribution.csv", index=False)
target_by_segment.to_csv(TABLES_DIR / "churn_risk_target_rate_by_kmeans_segment.csv", index=False)


# Final validation and exports

The final step checks that the columns needed for supervised modelling exist and then exports two main datasets:

1. a **full enriched dataset** for business review;
2. a **supervised-ready dataset** containing original behavioural/firmographic/segmentation features plus the rule-derived target.

Risk scores, status labels and recommended actions are excluded from the supervised feature file to reduce target leakage.

In [ ]:
# ============================================================
# 12. Validation and supervised-ready dataset construction
# ============================================================

key_output_cols = [
    "customer_risk_score", "customer_health_index", "customer_risk_class",
    "customer_status_label", "churn_risk_target", "recommended_action",
]

missing_outputs = [col for col in key_output_cols if col not in df.columns]
if missing_outputs:
    raise ValueError(f"Missing expected output columns: {missing_outputs}")

# Validate that the target has both classes; otherwise supervised learning would not be meaningful.
target_classes = df["churn_risk_target"].nunique()
if target_classes < 2:
    display(Markdown(
        "**Warning.** The rule-derived target has only one class in this dataset. "
        "Supervised classification will not be meaningful until the labeling rules or dataset are reviewed."
    ))
else:
    display(Markdown("**Validation passed.** The rule-derived target contains both classes and is usable for supervised classification."))

# Candidate features for the next supervised ML notebook.
# We include original behaviour, firmographics, geography/sector, and unsupervised outputs.
feature_candidates = []
feature_candidates += month_cols
feature_candidates += [
    "Revenue", "Employees", "total_volume", "recent_volume", "past_volume", "avg_monthly_volume",
    "volume_trend", "trend_pct", "volatility", "cv_volume", "recent_ratio", "activity_drop_ratio",
    "kmeans_cluster", "dbscan_label", "is_dbscan_noise",
    "Region", "Province", "NACE_Desc", "Nature",
]
feature_candidates = [col for col in feature_candidates if col in df.columns]

supervised_ready = df[feature_candidates + ["churn_risk_target"]].copy()

# This audit table keeps labels for interpretation but should not be used directly as model predictors.
audit_cols = [
    "kmeans_cluster", "kmeans_segment", "customer_risk_score", "customer_health_index",
    "customer_risk_class", "customer_status_label", "churn_risk_target",
    "value_at_risk_score", "business_priority_score", "recommended_action",
]
audit_cols = [col for col in audit_cols if col in df.columns]
modeling_audit = df[audit_cols].copy()

validation_summary = pd.DataFrame({
    "check": [
        "Customer records in final dataset",
        "Customer records in supervised-ready dataset",
        "Number of supervised predictors",
        "Target classes",
        "Missing values in target",
    ],
    "value": [
        len(df),
        len(supervised_ready),
        supervised_ready.drop(columns=["churn_risk_target"]).shape[1],
        target_classes,
        int(supervised_ready["churn_risk_target"].isna().sum()),
    ],
})

display(validation_summary)


In [ ]:
# ============================================================
# 13. Export final outputs
# ============================================================

full_output = OUTPUT_DIR / "dataset_profiled_risk_scored_status_labeled.csv"
supervised_output = OUTPUT_DIR / "dataset_supervised_ready_rule_based_target.csv"
profile_output = OUTPUT_DIR / "customer_segment_profile.csv"
risk_summary_output = OUTPUT_DIR / "customer_risk_summary_by_segment.csv"
tab_xi_output = OUTPUT_DIR / "tab_xi_risk_summary_by_kmeans_segment.csv"
status_summary_output = OUTPUT_DIR / "customer_status_summary.csv"
audit_output = OUTPUT_DIR / "customer_modeling_audit_labels.csv"
priority_output = OUTPUT_DIR / "top_priority_customers.csv"
target_by_segment_output = OUTPUT_DIR / "churn_risk_target_rate_by_segment.csv"
risk_mix_share_output = OUTPUT_DIR / "risk_class_composition_by_segment_share_pct.csv"

# Top customers for business action.
top_priority_cols = [
    "kmeans_segment", "customer_risk_class", "customer_status_label",
    "customer_risk_score", "customer_health_index", "value_score",
    "value_at_risk_score", "business_priority_score", "recommended_action",
]
top_priority_cols = [col for col in top_priority_cols if col in df.columns]
top_priority_customers = df.sort_values("business_priority_score", ascending=False)[top_priority_cols].head(50)

# Save files.
df.to_csv(full_output, index=False)
supervised_ready.to_csv(supervised_output, index=False)
segment_profile.to_csv(profile_output, index=False)
risk_summary_by_segment.to_csv(risk_summary_output, index=False)
if "tab_xi_risk_summary" in globals():
    tab_xi_risk_summary.round(2).to_csv(tab_xi_output, index=False)
status_distribution.to_csv(status_summary_output, index=False)
modeling_audit.to_csv(audit_output, index=False)
top_priority_customers.to_csv(priority_output, index=False)
if "target_by_segment" in globals():
    target_by_segment.to_csv(target_by_segment_output, index=False)
if "risk_mix" in globals():
    risk_mix.round(2).to_csv(risk_mix_share_output)

print("Saved outputs:")
for path in [
    full_output,
    supervised_output,
    profile_output,
    risk_summary_output,
    tab_xi_output,
    status_summary_output,
    audit_output,
    priority_output,
    target_by_segment_output,
    risk_mix_share_output,
]:
    print("-", path)

print("\nTop priority customers for review:")
display(top_priority_customers.round(2))


## Final outcome

This notebook completes the transition from unsupervised segmentation to supervised learning.

The final outputs provide:

- a compact profile of each K-Means segment;
- DBSCAN anomaly information integrated into risk interpretation;
- a transparent Customer Risk Score and Customer Health Index;
- three thesis-aligned risk classes: `Low Risk`, `Medium Risk`, `High Risk`;
- the risk-class composition by K-Means segment, saved as a thesis-ready figure;
- thesis-aligned customer status labels;
- a rule-derived `churn_risk_target` for supervised model training;
- status-label and target-readiness figures generated from the same final dataframe;
- a supervised-ready CSV that avoids using risk/status/action columns as predictors.

The next notebook can now train supervised models using `dataset_supervised_ready_rule_based_target.csv` as input.
